# 04 — The estimator: MLPClassifier and its parameters

You have built the whole machine by hand: a neuron, a hidden layer, and backpropagation. Now we drive
the real thing — scikit-learn's `MLPClassifier` — and meet **each knob from the concept that owns it**.
Nothing here is new magic; it is the network you already understand, with dials.

**Prerequisites**
- 11.1–11.3 — the neuron, the hidden layer, and backpropagation, built by hand.
- Chapter 03 — the loss curve and gradient descent; the softmax / cross-entropy for K classes.
- Chapter 00 — scaling with a `Pipeline` (fit on train only), over/underfitting and the
  generalization gap, cross-validation, hyperparameters vs parameters.

**What you'll be able to do**
- Drive `MLPClassifier`: pick the activation and solver, read the loss curve, size the network.
- Regularize with `alpha` (L2) and early stopping; scale features before fitting.
- Handle K classes with the softmax output head, and tune honestly with cross-validation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_blobs, make_circles, make_classification, make_moons
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
np.random.seed(SEED)

defaults = MLPClassifier().get_params()
for k in ["hidden_layer_sizes", "activation", "solver", "alpha",
          "learning_rate_init", "batch_size", "max_iter", "early_stopping"]:
    print(f"{k:20s} = {defaults[k]}")

## One object you already understand

`MLPClassifier` *is* the network from notebooks 11.1–11.3: a stack of layers, a forward pass, and
backpropagation driving gradient descent. The defaults above are the choices scikit-learn makes for
you — a single hidden layer of 100 units, ReLU activations, the Adam optimizer, a little L2
(`alpha`), and at most 200 passes over the data. The rest of this notebook takes those knobs one at a
time. One difference jumps out first: our by-hand network used the **sigmoid**, but the library
defaults to **ReLU**. The next cell shows why.

## Activation: why ReLU is the default

The sigmoid and tanh **saturate**: far from zero their curves flatten, so their slope — the gradient
that backprop multiplies by — shrinks toward zero. With the default mini-batch optimizer (Adam), a
sigmoid network can stall before it ever learns, stuck where the gradients are tiny. **ReLU**,
`max(0, z)`, has slope exactly 1 for positive inputs: no saturation there, so the gradient keeps
flowing. Let's watch the three activations train on the circles from notebook 11.2.

In [ ]:
Xc, yc = make_circles(n_samples=400, noise=0.10, factor=0.40, random_state=SEED)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.25, stratify=yc, random_state=SEED
)

C = colors.COLORS
act_color = {"logistic": C["class_a"], "tanh": C["class_b"], "relu": C["model"]}

fig, ax = plt.subplots(figsize=(8, 5))
for act in ["logistic", "tanh", "relu"]:
    m = MLPClassifier(hidden_layer_sizes=(16,), activation=act, solver="adam",
                      max_iter=500, random_state=SEED).fit(Xc_tr, yc_tr)
    ax.plot(m.loss_curve_, color=act_color[act], lw=2,
            label=f"{act}  (test acc {m.score(Xc_te, yc_te):.2f})")
    print(f"{act:9s}: train {m.score(Xc_tr, yc_tr):.3f}  test {m.score(Xc_te, yc_te):.3f}  "
          f"final loss {m.loss_:.3f}")
ax.axhline(np.log(2), color=C["muted"], ls=":", lw=1.5, label="ln 2 ≈ 0.69 (chance)")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("Same network on circles (solver=adam): sigmoid stalls, ReLU descends")
ax.legend()
plt.show()

**Read the figure.** With the default Adam optimizer the **sigmoid** network never leaves the
flat — its loss sits at `ln 2 ≈ 0.69` (the loss of always guessing 0.5) and accuracy is 0.39, a coin
flip. **tanh** does better but crawls; **ReLU** drops steadily and reaches accuracy 1.0. This is
exactly why notebooks 11.2–11.3 trained the by-hand sigmoid network with full-batch gradient descent
(and pinned `lbfgs` for the library) — sigmoid + mini-batch Adam saturates and stalls. Switching the
activation to ReLU removes the saturation, which is why **ReLU is the modern default**. (Sigmoid is
not broken — with `lbfgs` it reaches 1.0 too. The deeper story, vanishing gradients across *many*
layers, waits for chapter 12.)

**A note on the red warnings.** Running this cell prints a `ConvergenceWarning` — that is
information, not an error. It means a network hit its `max_iter` cap before the loss flattened. Here
it is expected: we capped training at 500 epochs, and the sigmoid never converges anyway. You will see
these whenever a network is still improving at the cap. The honest response is to raise `max_iter` (or
accept that the run was capped) — never to hide the warning.

## The optimizer, the loss curve, and epochs

The `solver` chooses *how* gradient descent is run. **`adam`** (the default) is an adaptive-step,
momentum-assisted upgrade of the plain descent you coded in 11.3 — it keeps a per-weight step size so
flat directions move faster and steep ones do not overshoot. **`sgd`** is plain mini-batch gradient
descent. **`lbfgs`** is a full-batch optimizer that shines on small datasets, but it does not expose a
per-epoch loss curve.

Three words set the rhythm of training. An **iteration** is one weight update on one **mini-batch**
(a subset of `batch_size` rows). An **epoch** is one full pass over the training set. `MLPClassifier`
records one `loss_curve_` point per **epoch**, and `max_iter` caps the number of epochs. The
`learning_rate_init` is the step size η from chapter 03. (A terminology heads-up: for these stochastic
solvers scikit-learn calls one *epoch* an "iteration", so `n_iter_` and `max_iter` count epochs, not
individual mini-batch updates.)

In [ ]:
print("solver on circles (relu, hidden=(16,), max_iter=2000):")
for solver in ["sgd", "adam", "lbfgs"]:
    m = MLPClassifier(hidden_layer_sizes=(16,), activation="relu", solver=solver,
                      max_iter=2000, random_state=SEED).fit(Xc_tr, yc_tr)
    has_curve = "yes" if hasattr(m, "loss_curve_") else "no"
    print(f"  {solver:6s}: train {m.score(Xc_tr, yc_tr):.3f}  test {m.score(Xc_te, yc_te):.3f}  "
          f"epochs {m.n_iter_:<5} loss_curve_ {has_curve}")

In [ ]:
# A consistent 2-D problem for the remaining knobs. From here we wrap every fit in a
# StandardScaler pipeline -- scaling is standard practice for a neural network (the section
# below shows why it is not optional), and a fitted Pipeline exposes the MLP via named_steps.
Xm, ym = make_moons(n_samples=300, noise=0.20, random_state=SEED)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    Xm, ym, test_size=0.25, stratify=ym, random_state=SEED
)


def fit_scaled(**kwargs):
    pipe = make_pipeline(StandardScaler(), MLPClassifier(random_state=SEED, **kwargs))
    return pipe.fit(Xm_tr, ym_tr)


print("learning_rate_init on moons (hidden=(50,), adam):")
for lr in [1e-4, 1e-3, 1e-2, 1e-1]:
    pipe = fit_scaled(hidden_layer_sizes=(50,), learning_rate_init=lr, max_iter=2000)
    mlp = pipe.named_steps["mlpclassifier"]
    print(f"  lr={lr:<7}: epochs {mlp.n_iter_:<5} final loss {mlp.loss_:.3f}  "
          f"test {pipe.score(Xm_te, ym_te):.3f}")

print()
print("batch_size on moons (hidden=(50,), adam, max_iter=500):")
for bs in [8, 32, len(Xm_tr)]:
    pipe = fit_scaled(hidden_layer_sizes=(50,), batch_size=bs, max_iter=500)
    mlp = pipe.named_steps["mlpclassifier"]
    print(f"  batch_size={str(bs):4s}: epochs (n_iter_) {mlp.n_iter_:<5} "
          f"len(loss_curve_) {len(mlp.loss_curve_):<5} test {pipe.score(Xm_te, ym_te):.3f}")

**Read the tables.** All three solvers reach the same accuracy on this easy problem, but they get
there differently: Adam converges in fewer epochs than plain SGD, and `lbfgs` finishes in ~20 steps —
yet only the mini-batch solvers expose a `loss_curve_` to watch. For the learning rate, too small
(`1e-4`) crawls for many epochs and underfits (test 0.87); the default and larger rates converge fast.
And the batch table confirms the vocabulary: **`len(loss_curve_)` equals `n_iter_` equals the number
of epochs** — one recorded point per full pass, whatever the batch size.

## Capacity: `hidden_layer_sizes`

`hidden_layer_sizes` sets the network's shape — `(50,)` is one hidden layer of 50 units, `(50, 50)`
is two. More units and more layers mean a more intricate boundary the network *can* draw. Too few
units and it cannot bend enough to fit the data (underfitting); plenty of capacity is then kept in
check by regularization (next section). Let's see the extremes on the moons.

In [ ]:
small = fit_scaled(hidden_layer_sizes=(1,), max_iter=3000)
big = fit_scaled(hidden_layer_sizes=(50,), max_iter=3000)
print(f"hidden=(1,) : train {small.score(Xm_tr, ym_tr):.3f}  test {small.score(Xm_te, ym_te):.3f}")
print(f"hidden=(50,): train {big.score(Xm_tr, ym_tr):.3f}  test {big.score(Xm_te, ym_te):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
viz.plot_decision_boundary(small, Xm_te, ym_te, ax=axes[0])
axes[0].set_title(f"One hidden unit (test {small.score(Xm_te, ym_te):.2f})")
viz.plot_decision_boundary(big, Xm_te, ym_te, ax=axes[1])
axes[1].set_title(f"50 hidden units (test {big.score(Xm_te, ym_te):.2f})")
plt.show()

**Read the figure.** A single ReLU hidden unit can draw only one fold — it cannot wrap around both
crescents, so it underfits, catching only about 0.85 of the points with a near-straight boundary.
Fifty units have the capacity to carve the two interlocking moons and reach 0.97. Width and depth are
the capacity dial: turn it up until the network can represent the shape, then control any surplus with
regularization.

## Regularization: `alpha` and early stopping

`alpha` is the strength of the **L2 penalty** on the weights (weight decay, chapter 03 NB 5): larger
`alpha` pulls weights toward zero and smooths the boundary. **Early stopping** is a different guard —
set `early_stopping=True` and the network holds out `validation_fraction` of the training data, watches
the score on it each epoch, and halts when it stops improving. Let's sweep `alpha` and then toggle
early stopping on a high-capacity network.

In [ ]:
print("alpha (L2) on moons (hidden=(200, 200), adam):")
for a in [1e-6, 1e-2, 1e-1, 1.0, 10.0]:
    pipe = fit_scaled(hidden_layer_sizes=(200, 200), alpha=a, max_iter=3000)
    tr, te = pipe.score(Xm_tr, ym_tr), pipe.score(Xm_te, ym_te)
    print(f"  alpha={a:<7}: train {tr:.3f}  test {te:.3f}")

print()
print("early_stopping on moons (hidden=(200, 200), adam):")
for es in [False, True]:
    pipe = fit_scaled(hidden_layer_sizes=(200, 200), early_stopping=es, max_iter=3000)
    mlp = pipe.named_steps["mlpclassifier"]
    te = pipe.score(Xm_te, ym_te)
    tag = f"  best_validation_score_ {mlp.best_validation_score_:.3f}" if es else ""
    print(f"  early_stopping={es!s:5}: epochs {mlp.n_iter_:<5} test {te:.3f}{tag}")

**Read the tables.** A small `alpha` fits well (train 0.98, test 0.97) with a hair of overfitting; a
touch more — around `alpha=0.1` — is the sweet spot here (test 0.99). Push it further and the penalty
starts erasing the weights: at `alpha=1` accuracy slips (0.91), and at `alpha=10` the boundary is
smoothed into underfitting (0.81). Too little or too much L2 both cost you. Early stopping, meanwhile,
halted after about a dozen epochs and gave up accuracy — it spends a slice of training data on a
validation watch and stops early. It earns its keep on large, long-training problems where overfitting
is real; on a small clean set like this it is not a free win. The right amount of either guard depends
on how much your data tempts the model to overfit.

## Scaling is not optional for neural networks

Gradient descent steps every weight by the same learning rate. If one feature runs over a far larger
range than another, its weights see far larger gradients, the loss surface becomes a stretched ravine,
and descent crawls. Putting the features on a common scale first — a `StandardScaler`, **fit on the
training data only** (chapter 00) — fixes this. Here is the same network on a two-feature problem where
the second feature has been multiplied by 100.

In [ ]:
Xs, ys = make_classification(n_samples=600, n_features=2, n_informative=2, n_redundant=0,
                             n_clusters_per_class=1, class_sep=1.2, random_state=SEED)
Xs[:, 1] *= 100.0   # a deliberate scale mismatch between the two features
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
    Xs, ys, test_size=0.25, stratify=ys, random_state=SEED
)

unscaled = MLPClassifier(
    hidden_layer_sizes=(100,), max_iter=300, random_state=SEED
).fit(Xs_tr, ys_tr)
scaled = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=SEED),
).fit(Xs_tr, ys_tr)
scaled_mlp = scaled.named_steps["mlpclassifier"]
print(f"unscaled: test {unscaled.score(Xs_te, ys_te):.3f}  final loss {unscaled.loss_:.3f}")
print(f"scaled  : test {scaled.score(Xs_te, ys_te):.3f}  final loss {scaled_mlp.loss_:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(unscaled.loss_curve_, color=C["error"], lw=2, label="unscaled features")
ax.plot(scaled_mlp.loss_curve_, color=C["model"], lw=2, label="StandardScaler + MLP")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss (log scale)")
ax.set_title("Why scaling matters: unscaled features start high and crawl")
ax.legend()
plt.show()

**Read the figure.** The unscaled network starts with a loss near 7 and limps downward, reaching only
test accuracy 0.94 in its 300 epochs. The scaled pipeline starts near 0.7 and converges fast and low,
reaching 0.99. Same data, same network — the only difference is a `StandardScaler` in front. For a
neural network, scaling is part of the model: always wrap it in a `Pipeline(StandardScaler, MLP)` so
the scaler is fit on the training fold only and never peeks at the test data.

## From one output to K: the softmax head

Every network so far had a single output unit with a sigmoid — perfect for two classes. For **K**
classes the output layer grows to **K units**, and the sigmoid is replaced by the **softmax**, which
turns the K raw scores into K probabilities that are all positive and **sum to 1** (chapter 03 NB 5).
The loss becomes the K-class cross-entropy. `MLPClassifier` builds this head automatically: give it a
target with three labels and it grows three output units.

In [ ]:
Xb, yb = make_blobs(n_samples=600, centers=3, n_features=2, cluster_std=1.2, random_state=SEED)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    Xb, yb, test_size=0.25, stratify=yb, random_state=SEED
)
mb = MLPClassifier(hidden_layer_sizes=(20,), max_iter=1000, random_state=SEED).fit(Xb_tr, yb_tr)
print(f"n_outputs_ = {mb.n_outputs_}   out_activation_ = {mb.out_activation_}   "
      f"test acc {mb.score(Xb_te, yb_te):.3f}")
proba = mb.predict_proba(Xb_te[:3])
print("predict_proba (first 3 rows):")
print(np.round(proba, 4))
print("row sums:", np.round(proba.sum(axis=1), 6).tolist())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axis("off")
hidden = [(1.0, hy) for hy in (2.6, 1.8, 1.0)]
outs = [(3.4, 2.6, "P(class 0)"), (3.4, 1.6, "P(class 1)"), (3.4, 0.6, "P(class 2)")]
for hx, hy in hidden:
    for ox, oy, _ in outs:
        ax.plot([hx, ox], [hy, oy], color=C["muted"], lw=0.7, zorder=1)
for hx, hy in hidden:
    ax.add_patch(plt.Circle((hx, hy), 0.16, color=C["class_c"], ec=C["text"], lw=1.2, zorder=3))
for ox, oy, label in outs:
    ax.add_patch(plt.Circle((ox, oy), 0.18, color=C["class_b"], ec=C["text"], lw=1.2, zorder=3))
    ax.text(ox + 0.35, oy, label, va="center", color=C["text"], fontsize=10)
ax.text(1.0, 3.1, "hidden layer", ha="center", color=C["muted"], fontsize=10)
ax.text(3.4, 3.1, "K = 3 softmax outputs", ha="center", color=C["muted"], fontsize=10)
ax.text(3.4, 0.0, "softmax → sums to 1", ha="center", color=C["highlight"], fontsize=10)
ax.set_xlim(0.3, 5.2)
ax.set_ylim(-0.4, 3.4)
ax.set_title("The K-class output head")
plt.show()

**Read the figure.** The hidden layer feeds **K = 3** output units, one per class, and the softmax
normalizes their scores into a probability vector — each row of `predict_proba` is positive and sums
to exactly 1. The binary sigmoid you have used all along is the K = 2 special case of this same head.
Nothing else about the network changes: same layers, same backpropagation, a wider output.

In [ ]:
# Now that scaling is part of the model, tune inside a pipeline with cross-validation.
default = make_pipeline(
    StandardScaler(), MLPClassifier(max_iter=2000, random_state=SEED)
).fit(Xm_tr, ym_tr)

param_grid = {
    "mlpclassifier__hidden_layer_sizes": [(10,), (50,), (50, 50)],
    "mlpclassifier__alpha": [1e-4, 1e-2, 1.0],
    "mlpclassifier__learning_rate_init": [1e-3, 1e-2],
}
search = GridSearchCV(
    make_pipeline(StandardScaler(), MLPClassifier(max_iter=2000, random_state=SEED)),
    param_grid, cv=5, n_jobs=-1,
).fit(Xm_tr, ym_tr)

print(f"default test accuracy = {default.score(Xm_te, ym_te):.3f}")
print(f"tuned   test accuracy = {search.score(Xm_te, ym_te):.3f}")
print(f"best params = {search.best_params_}")

**Read the result.** The defaults are already a strong baseline. Cross-validated grid search bought a
small, genuine improvement here (a point or two), choosing a compact network with a slightly larger
learning rate. The honest number is the **one sealed test** at the end — the grid was scored by
cross-validation on the *training* data, and only the winner touches the test set once. Tune when you
have a reason and a budget; do not read the leaderboard of fold scores as your final accuracy.

## Your turn

1. **(warm-up)** Re-fit the circles network with `activation="logistic"` and the default `adam`, and
   predict the loss curve before running. Then add `solver="lbfgs"` and watch it recover — why does
   the solver rescue the sigmoid?
2. **(core)** On the moons, set `hidden_layer_sizes=(300, 300)` with `alpha=1e-6`, then raise `alpha`
   step by step. Where does the test score peak, and where does too much L2 start to hurt?
3. **(reach)** Fit a network with `early_stopping=True` and plot its `validation_scores_` on the same
   axes as its `loss_curve_`. What is each one measuring, and why can the validation score turn while
   the training loss is still falling?

## What you built

- You drove `MLPClassifier` and saw why **ReLU** is the default — sigmoid saturates and stalls under
  Adam.
- You read the **loss curve** and named the optimizer's rhythm: iteration, mini-batch, **epoch**
  (`len(loss_curve_)`).
- You sized a network with `hidden_layer_sizes`, regularized it with `alpha` and early stopping, and
  made **scaling** part of the model.
- You met the **K-class softmax head** (probabilities summing to 1), and tuned honestly with
  cross-validation against one sealed test.

Next: the capstone — handwritten digits, end to end, where all of this meets a real multi-class
problem.

## References

- Kingma, D. P. & Ba, J. (2015). *Adam: A Method for Stochastic Optimization.* ICLR.
  https://arxiv.org/abs/1412.6980
- Nair, V. & Hinton, G. E. (2010). *Rectified Linear Units Improve Restricted Boltzmann Machines.*
  ICML — the ReLU activation.
- Glorot, X. & Bengio, Y. (2010). *Understanding the difficulty of training deep feedforward neural
  networks.* PMLR 9, 249–256 — saturation and initialization.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*, §5 (network training). Springer.
- scikit-learn user guide: *Neural network models (supervised)* — `MLPClassifier`.

*Previous:* 11.3 — backpropagation.  *Next:* **11.5 — the handwritten-digits capstone.**